<h1>Sarcasm detection system using custom
Naïve Bayes implementations</h1>

In [90]:
import numpy as np
import pandas as pd
import pandas as pd
import matplotlib.pyplot as plt
import re
import json
import math 
from collections import defaultdict, Counter
from sklearn.metrics  import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


In [91]:
student_ID = 276690
last_6     = student_ID % 1000000
seed       = last_6 % 100000

In [92]:
df = pd.read_json('/Users/arjun/Downloads/Data/Sarcasm_Headlines_Dataset.json', lines=True)
print(df.shape)
df.head()



(26709, 3)


,article_link,headline,is_sarcastic
0,https://www.huffingtonpost.com/entry/versace-b...,former versace store clerk sues over secret 'b...,0
1,https://www.huffingtonpost.com/entry/roseanne-...,the 'roseanne' revival catches up to our thorn...,0
2,https://local.theonion.com/mom-starting-to-fea...,mom starting to fear son's web series closest ...,1
3,https://politics.theonion.com/boehner-just-wan...,"boehner just wants wife to listen, not come up...",1
4,https://www.huffingtonpost.com/entry/jk-rowlin...,j.k. rowling wishes snape happy birthday in th...,0


In [93]:
#shuffling
df_shuffled = df.sample(frac=1, random_state=seed)
df_shuffled.head()

,article_link,headline,is_sarcastic
19914,https://www.theonion.com/bus-stop-ad-has-more-...,bus-stop ad has more legal protections than av...,1
17336,https://www.theonion.com/study-finds-68-of-ame...,study finds 68% of americans unprepared for su...,1
9650,https://politics.theonion.com/mute-terrified-r...,"mute, terrified rubio awakes to find self unab...",1
18451,https://www.huffingtonpost.com/entry/viceland-...,two trans women from the bronx open up about t...,0
20784,https://www.theonion.com/nipple-of-baby-s-bott...,nipple of baby's bottle pierced for authenticity,1


In [94]:
#70/15/15 split 
n = len(df_shuffled)
training_upper_lim = int(0.7 * n)
validation_upper_lim = int(0.85 * n)
training_dataset = df_shuffled.iloc[: training_upper_lim]
validation_dataset = df_shuffled.iloc[training_upper_lim:validation_upper_lim]
test_dataset = df_shuffled.iloc[validation_upper_lim:]

In [95]:
print(n)
print(f'train dataset count : {len(training_dataset)}')
print(f'validation dataset count : {len(validation_dataset)}')
print(f'test dataset count : {len(test_dataset)}')
total = len(training_dataset) + len(validation_dataset) + len(test_dataset)
print(f'{'count verified'if total ==n else 'mismatch in count'}')




26709
train dataset count : 18696
validation dataset count : 4006
test dataset count : 4007
count verified


In [96]:
#smoothing
smoothing_sets = {
    0: [0, 0.1, 1],
    1: [0, 0.5, 2],
    2: [0, 1,   5],
}
k = student_ID % 4 

**Task 1 – Tokenisation & Preprocessing**

In [97]:
#basic cleaning
def basic_cleaning (text):
     text = text.lower()
     text = re.sub(r"<.*?>", "", text) #html
     text = re.sub(r'!{2,}',    '[EXC_SEQ]',  text) #Exclamation Sequences: !!, !!!, etc. à[EXC_SEQ]
     text = re.sub(r'\?{2,}',   '[QUE_SEQ]',  text)  #Question Sequences: ??, ???, etc. à[QUE_SEQ]
     text = re.sub(r'[?!]{2,}', '[INT_MARK]', text)  #Interrobangs: Any combination of ?! or !? à[INT_MARK]
     text = re.sub(r'\.{3,}',   '[ELLIP]',    text) #Ellipses: ... à [ELLIP]

     text = re.sub(r"[^\w\s']",       '', text)   # remove non word/space/apostrophe
     text = re.sub(r"(?<!\w)'|'(?!\w)", '', text)  # remove apostrophes NOT inside words

     text = re.findall(r'\[.*?\]|\w+(?:\'\w+)*', text)
     return text
     

references:
https://www.geeksforgeeks.org/python/how-to-remove-html-tags-from-string-in-python/

https://regex101.com/

https://docs.python.org/3/library/re.html#re.sub #pattern substitution


In [98]:
df_shuffled['headline'].iloc[1]

'study finds 68% of americans unprepared for sudden financial stability'

In [99]:
#testing regex
sample_headlines = [
    "Scientists DISCOVER!!! That Water Is WET!!",   
    "Are? you?? serious????",             
    "Oh reaLly?! come on!!??",      
    "just waiting...",    
    "<b>Breaking news</b>: bomb blast", 
    "senate's new plan to [repeal] but not replace",  
    "a 'damn' letter from a serial killer",    
]

data=[]
for i in sample_headlines:
    word = basic_cleaning(i)
    print(word)




['scientists', 'discoverEXC_SEQ', 'that', 'water', 'is', 'wetEXC_SEQ']
['are', 'youQUE_SEQ', 'seriousQUE_SEQ']
['oh', 'reallyINT_MARK', 'come', 'onEXC_SEQQUE_SEQ']
['just', 'waitingELLIP']
['breaking', 'news', 'bomb', 'blast']
["senate's", 'new', 'plan', 'to', 'repeal', 'but', 'not', 'replace']
['a', 'damn', 'letter', 'from', 'a', 'serial', 'killer']


In [114]:
#Negation Handling
negation_words={ "not", "no", "never", "don't", "didn't", "isn't", "wasn't"}
stop_words={"the", "a","also","an","and","are","as","at","be","because","been","but",
            "by","for","from","has","have","however","if","in","is","not","of",
            "on","or","so","than","that","their","there","these","this","to","was","were"}
vowels = set('aeiou')

def count_vowels(token):
    return sum(1 for ch in token if ch in vowels )
    
def negation_handling(tokens):
    result = []
    i = 0
    while i < len(tokens):
        token = tokens[i]
        if token in negation_words:
            result.append(token)
            n_vowels = count_vowels(token)
            i += 1
            for _ in range(n_vowels):
                if i < len(tokens):
                    result.append("NOT_" + tokens[i])
                    i += 1
        else:
            result.append(token)
            i += 1
    return result
    


In [101]:

def stopword_removal(tokens):
    return [t for t in tokens if t.startswith("NOT_") or t not in stop_words]
 
 
def generate_ngrams(tokens):
    unigrams = tokens
    bigrams  = [f"{tokens[i]}_{tokens[i+1]}" for i in range(len(tokens) - 1)]
    return unigrams + bigrams
 

In [102]:
def replace_oov(tokens, vocab):
    return [t if t in vocab else "<UNK>" for t in tokens]

In [103]:
def preprocess(text, vocab=None, ablation_k=None):
    tokens = basic_cleaning(text)

    if ablation_k != 2:
        tokens = negation_handling(tokens)

    if ablation_k != 3:
        tokens = stopword_removal(tokens)

    all_tokens = generate_ngrams(tokens)

    if ablation_k == 0:
        all_tokens = [t for t in all_tokens if "_" in t]
    elif ablation_k == 1:
        all_tokens = [t for t in all_tokens if "_" not in t]

    if vocab is not None:
        all_tokens = replace_oov(all_tokens, vocab)

    return all_tokens

In [113]:
#OOV Handling 

#building vocab _ training data
def build_vocabulary(training_dataset, preprocessor):
    vocab = set()
    for headline in training_dataset['headline']:
        tokens = preprocessor.preprocess(headline)
        vocab.update(tokens)
    return vocab

train_tokens = training_dataset["headline"].apply(lambda x: preprocess(x))
vocab            = set(tok for doc in train_tokens for tok in doc)
 
train_tokens = training_dataset["headline"].apply(lambda x: preprocess(x, vocab))
val_tokens   = validation_dataset["headline"].apply(lambda x: preprocess(x, vocab))
test_tokens  = test_dataset["headline"].apply(lambda x: preprocess(x, vocab))

In [110]:


val_tokens.head()


19829    [trump, shifts, infrastructure, james, comey, ...
16770    [<UNK>, <UNK>, gets, laugh, out, pushing, <UNK...
11338    [monopoly, releases, special, regular, monopol...
6810     [trump, privately, terrified, his, sexual, ass...
12925    [artists, explore, many, influences, <UNK>, je...
Name: headline, dtype: object

In [109]:
test_tokens.head()

10498    [white, house, blocks, <UNK>, <UNK>, white_hou...
5157     [fan, <UNK>, world, series, home, run, another...
21408    [lax, petsmart, background, check, allows, der...
11724    [man, tentatively, takes, shot, <UNK>, girlfri...
21774    [brad, pitt, sidelined, 6, 8, weeks, with, red...
Name: headline, dtype: object

In [111]:
train_tokens.head()

19914    [busstop, ad, more, legal, protections, averag...
17336    [study, finds, 68, americans, unprepared, sudd...
9650     [mute, terrified, rubio, awakes, find, self, u...
18451    [two, trans, women, bronx, open, up, about, li...
20784    [nipple, baby's, bottle, pierced, authenticity...
Name: headline, dtype: object

<h1>Task 2 – Manual Feature Representation</h1>

In [ ]:
#Count Vectoriser
t = (seed % 6) + 2 #this clean up the noise, any word appearing less than t value will be removed

#vocabulary sorting:


In [ ]:
#Compute Document Frequency
def compute_doc_freq(tokenised_doc):
    doc_freq_count ={}
    for doc in tokenised_doc:
        for tkn in set(doc): #to remove th duplicates
            doc_freq_count[tkn]= doc_freq_count.get(tkn,0) + 1

    return doc_freq_count
